# ViSceT5 + TWC — Finetune + Ablation · nhánh `exp/twc-ocr-centric` · Colab

Chạy **đúng quy trình của repo**: notebook chỉ đặt biến môi trường rồi gọi `bash run_all_colab.sh finetune`
(clone nhánh → `setup.sh` → `run_pipeline.py` → `training/finetune.py configs/finetune.yaml` → push checkpoint mỗi epoch lên HF).
Mọi thông số huấn luyện lấy từ **`configs/finetune.yaml`** (5 epoch, batch 8 × tích luỹ gradient 2 = batch hiệu dụng 16, LR 3e-5 cosine, không label smoothing, beam 4, chọn checkpoint theo (EM+F1)/2 trên val); notebook **không** ghi đè.
Module nào tắt thì **không được dựng** (không tham số). Loss = VQA + λ·TWC khi bật TWC (λ = `twc_weight` trong yaml).

Mỗi lần chạy = **một cấu hình**, chọn bằng các công tắc bật/tắt module:

| Công tắc | `True` | `False` |
|---|---|---|
| `USE_IMAGE` | có đặc trưng ảnh (CLIP) | **bỏ hẳn ảnh**: chỉ câu hỏi + OCR (CLIP không chạy) |
| `USE_QAVIT` | đưa câu hỏi vào CLIP (QA-ViT) | CLIP trích xuất ảnh thường (không câu hỏi) |
| `USE_SCESPAVIS` | OCR qua SceSpaVis (ConsFormer + char + box + det/rec) | OCR text-only |
| `USE_OCRAUG` | nối thêm bản OCR đã sửa (kênh sửa OCR) | chỉ OCR gốc |
| `USE_TWC` | thêm loss TWC (TWA): train dùng tăng cường TWA (p=0.35, nhãn 1.0/0.9), eval/test dùng bản sửa xác định | không TWC |
| `TWC_QUEUE` | số đặc trưng từ của các batch trước giữ trong bộ nhớ làm negative thêm (mặc định 16384 ≈ 55 lượt forward gần nhất) | `0`: chỉ so trong batch (~300 ứng viên) |

TWC dùng softmax InfoNCE trên đặc trưng chữ của OCR (text ConsFormer + ký tự); ứng viên trùng chữ với từ đang xét không bị coi là negative. Bộ nhớ chỉ được đọc sau `twc_queue_start` bước (yaml), không lưu vào checkpoint.

**ViSceT5 + TWC:** `USE_IMAGE=True, USE_QAVIT=False, USE_SCESPAVIS=True, USE_OCRAUG=True, USE_TWC=True`. `USE_TWC` cần `USE_OCRAUG=True`.

## 1. Lấy repo (nhánh `exp/twc-ocr-centric`)

In [ ]:
import os
BRANCH = "exp/twc-ocr-centric"
if not os.path.exists("/content/ViSceT5"):
    !git clone -b {BRANCH} https://github.com/Kussssssss/ViSceT5.git /content/ViSceT5
%cd /content/ViSceT5
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -3
assert os.path.exists("data/ocr_correction.py") and "def _twc_memory" in open("models/openvivqa_model.py", encoding="utf-8").read(), "Sai nhánh hoặc code cũ — cần exp/twc-ocr-centric có bộ nhớ TWC!"

## 2. Hugging Face token (để `finetune.py` tự push checkpoint mỗi epoch, tự resume khi Colab ngắt, và upload cuối)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 3. Cấu hình — bật/tắt module

In [ ]:
USE_IMAGE     = True
USE_QAVIT     = False
USE_SCESPAVIS = True
USE_OCRAUG    = True
USE_TWC       = True
TWC_QUEUE     = 16384
DATASET       = "ViTextVQA"
RUN_TAG       = "t4"
PREDICT_CKPT  = "best"

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

assert USE_IMAGE or not USE_QAVIT, "QA-ViT cần ảnh: khi USE_IMAGE=False phải đặt USE_QAVIT=False"
assert USE_OCRAUG or not USE_TWC, "TWC cần nửa OCR related: khi USE_OCRAUG=False phải đặt USE_TWC=False"
RUN_NAME = "-".join(m for m, on in (("noimg", not USE_IMAGE), ("qavit", USE_QAVIT), ("scespavis", USE_SCESPAVIS),
                                    ("ocraug", USE_OCRAUG), (f"twc-q{TWC_QUEUE}" if TWC_QUEUE else "twc", USE_TWC)) if on) or "base"
OUTPUT_DIR = f"/content/ViSceT5/output/twc_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-twcbr-{DATASET.lower()}-{RUN_NAME}-{RUN_TAG}"
_b = lambda v: "true" if v else "false"

os.environ.update({
    "GIT_BRANCH":           "exp/twc-ocr-centric",
    "ABLATION_USE_IMAGE":   _b(USE_IMAGE),
    "ABLATION_USE_QACLIP":  _b(USE_QAVIT),
    "ABLATION_USE_OCR":     _b(USE_SCESPAVIS),
    "ABLATION_USE_OCR_AUG": _b(USE_OCRAUG),
    "USE_TWC":              _b(USE_TWC),
    "TWC_QUEUE_SIZE":       str(int(TWC_QUEUE)),
    "DATASET_NAME":         DATASET,
    "OUTPUT_DIR":           OUTPUT_DIR,
    "HF_REPO":              HF_REPO,
})
_main = USE_IMAGE and (not USE_QAVIT) and USE_SCESPAVIS and USE_OCRAUG
print(f"Cấu hình: Ảnh={USE_IMAGE} | QA-ViT={USE_QAVIT} | SceSpaVis={USE_SCESPAVIS} | OCR-Aug={USE_OCRAUG} | TWC={USE_TWC}"
      f"{' (bộ nhớ ' + str(TWC_QUEUE) + ')' if USE_TWC else ''}"
      f"{'   ← ViSceT5' + (' + TWC' if USE_TWC else '') if _main else ''}")
print(f"RUN_NAME={RUN_NAME}\nOUTPUT_DIR={OUTPUT_DIR}\nHF_REPO={HF_REPO}")
print(f"PREDICT_CKPT={PREDICT_CKPT}")
print("Thông số huấn luyện: lấy từ configs/finetune.yaml")

## 4. (Tuỳ chọn) Mock test — vài bước để bắt lỗi trước khi chạy thật
Dùng chế độ `mock` của `run_all_colab.sh` (`--smoke_test`: 8 mẫu train, 4 mẫu val, 3 bước); không push, không resume từ HF. `TWC_QUEUE_START=1` để bộ nhớ TWC được đọc ngay từ bước 2 — kiểm tra luôn nhánh bộ nhớ trên GPU (bản thật chỉ đọc sau 1000 bước).

In [ ]:
!HF_REPO= OUTPUT_DIR={OUTPUT_DIR}_mock TWC_QUEUE_START=1 bash run_all_colab.sh finetune mock

## 5. Finetune (từ đầu, không pretrain) — log train có thêm `twc_loss`, `twc_acc` khi bật TWC

In [ ]:
!bash run_all_colab.sh finetune

## 6. Dự đoán dev + test (`STAGE=predict` của `run_pipeline.py`) và chấm điểm
`predict.py` in EM/F1 trên dev và ghi file nộp cho test. File được chép vào `OUTPUT_DIR` để không đè giữa các cấu hình.
Test được chấm bằng `training/benchmark_eval.py` — **đúng giao thức benchmark** (cùng cách chấm các dòng baseline/LMM trong bài): EM, F1 token/char, BLEU-1..4/avg, CIDEr, BERTScore (mBERT).
Đáp án test lấy từ `eval_gt/<DATASET>_test_gt.json` trong repo (đã có `ViTextVQA`); dataset chưa có file thì cell hiện nút upload **ngay đầu cell** (trước khi predict).
Nếu `OUTPUT_DIR` (hoặc `predictions/` trên HF) đã có `submission_test.csv` thì cell **bỏ qua predict** và chỉ chấm điểm.
File nộp, log predict (có EM/F1 dev) và kết quả chấm test được upload vào thư mục `predictions/` của repo HF của cấu hình.

**Chọn checkpoint:** `PREDICT_CKPT` ở cell 3. Cell in bảng các checkpoint kèm EM/F1/CIDEr trên val của từng epoch rồi dùng checkpoint được chọn:
`best` = model tốt nhất (thư mục gốc, do `trainer.save_model` lưu sau `load_best_model_at_end`), `latest`, `epoch3`, hoặc `checkpoint-17580`.
Chỉ tải file inference của checkpoint đó (không tải optimizer). Kết quả của `best` nằm ở `OUTPUT_DIR` và `predictions/` trên HF;
của checkpoint khác nằm ở `OUTPUT_DIR/predict_checkpoint-N` và `predictions/checkpoint-N/` → không đè nhau.

**Predict lại sau khi Colab đã tắt:** chạy cell 1, 2, 3 (đặt đúng công tắc, `RUN_TAG` của lần train và `PREDICT_CKPT`) rồi chạy thẳng cell này, bỏ qua mock và finetune.

In [ ]:
import os, shutil, glob
!git pull -q origin {BRANCH}
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    TEST_GT_JSON = f"/content/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    from google.colab import files
    print(f"⬆️ Chọn file {os.path.basename(TEST_GT_JSON)} để upload (bấm Cancel nếu chưa có → vẫn predict, chưa chấm test)")
    up = files.upload()
    if up:
        shutil.move(os.path.abspath(next(iter(up))), TEST_GT_JSON)
print("GT test:", TEST_GT_JSON if os.path.exists(TEST_GT_JSON) else "chưa có")

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, PRED_DIR, PRED_REPO_PATH = CKPT_INFO["model_dir"], CKPT_INFO["pred_dir"], CKPT_INFO["pred_path_in_repo"]
assert os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")), f"Không có model trong {MODEL_DIR}"

tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
pred = os.path.join(PRED_DIR, f"submission_{tag}test.csv")
if os.path.exists(pred):
    print(f"✅ Đã có {pred} → bỏ qua predict, chỉ chấm điểm.")
else:
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{tag}{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{tag}{sp}.csv")
    !STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={DATASET} PREDICT_CKPT_DIR={MODEL_DIR} HF_REPO= bash run_all_colab.sh
    for sp in ("dev", "test"):
        src = f"/content/ViSceT5/output/submission_{tag}{sp}.csv"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(PRED_DIR, os.path.basename(src)))
            print("→", os.path.join(PRED_DIR, os.path.basename(src)))
    shutil.copy2("/content/ViSceT5/train_execution.log", os.path.join(PRED_DIR, "predict_log.txt"))
log = os.path.join(PRED_DIR, "predict_log.txt")
dev_lines = [l.strip() for l in open(log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l] if os.path.exists(log) else []
if dev_lines:
    print(dev_lines[-1])
if os.path.exists(TEST_GT_JSON) and os.path.exists(pred):
    !pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap
    !python -m training.benchmark_eval --gt {TEST_GT_JSON} --pred {pred} --out {PRED_DIR}/eval_test.json
else:
    print("ℹ️ Chưa đặt TEST_GT_JSON → chưa chấm test.")

if os.environ.get("HF_TOKEN") and HF_REPO:
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {PRED_DIR} --path_in_repo {PRED_REPO_PATH} --files submission_{tag}dev.csv submission_{tag}test.csv predict_log.txt eval_test.json
else:
    print("⚠️ Không có HF_TOKEN/HF_REPO → file dự đoán chỉ nằm trên Colab, hãy tải về trước khi ngắt phiên.")

## 7. Tổng hợp kết quả mọi cấu hình đã chạy

In [ ]:
import json, os, glob
import pandas as pd
rows = []
for od in sorted(glob.glob(f"/content/ViSceT5/output/twc_{DATASET}_*")):
    name = od.split(f"twc_{DATASET}_")[-1]
    mods = set(name.split("-"))
    r = {"run": name, "Ảnh": "noimg" not in mods, "QA-ViT": "qavit" in mods,
         "SceSpaVis": "scespavis" in mods, "OCR-Aug": "ocraug" in mods, "TWC": "twc" in mods}
    st = sorted(glob.glob(os.path.join(od, "checkpoint-*", "trainer_state.json")),
                key=lambda p: int(p.split("checkpoint-")[-1].split(os.sep)[0]))
    if st:
        hist = json.load(open(st[-1])).get("log_history", [])
        key = "eval_cider" if DATASET == "OpenViVQA" else "eval_em_f1"
        evs = [h for h in hist if "eval_em" in h]
        if evs:
            best = max(evs, key=lambda h: h.get(key, 0.5 * (h.get("eval_em", 0) + h.get("eval_f1", 0))))
            r.update({"val_EM": round(100 * best.get("eval_em", float("nan")), 2), "val_F1": round(100 * best.get("eval_f1", 0), 2),
                      "val_CIDEr": round(best.get("eval_cider", float("nan")), 4), "best_epoch": best.get("epoch")})
    for pd_dir in [od] + sorted(glob.glob(os.path.join(od, "predict_checkpoint-*"))):
        rr = dict(r, ckpt="best" if pd_dir == od else os.path.basename(pd_dir)[len("predict_"):])
        ev = os.path.join(pd_dir, "eval_test.json")
        if os.path.exists(ev):
            t = json.load(open(ev, encoding="utf-8"))
            rr.update({"test_EM": round(100 * t.get("Exact Match (EM)", float("nan")), 2), "test_F1": round(100 * t.get("F1 (Token-level)", float("nan")), 2),
                       "test_BLEU4": round(t.get("BLEU-4", float("nan")), 4), "test_CIDEr": round(t.get("CIDEr", float("nan")), 4),
                       "test_BERTScore": round(100 * t.get("BERTScore_F1", float("nan")), 2)})
        elif pd_dir != od:
            continue
        rows.append(rr)
df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "Chưa có cấu hình nào chạy xong.")

## 8. Trực quan hoá attention (cấu hình hiện tại)
`scripts/visualize_attention.py` sinh đáp án (beam search như predict) rồi lấy **cross-attention** của các token đáp án lên đầu vào encoder,
trung bình trên mọi lớp và mọi head. Mỗi câu hỏi ra một hình: heatmap trên 14×14 patch ảnh (nếu mô hình dùng ảnh), box OCR tô màu theo attention,
và attention trên từng từ của câu hỏi, OCR gốc, OCR đã sửa (viền đỏ = từ bị sửa). `segment_mass.csv` ghi tỷ lệ attention dồn vào từng phần.
Dùng checkpoint `PREDICT_CKPT` (cell 3). Đặt `ATTN_IDS` để chọn câu hỏi cụ thể; để trống thì lấy ngẫu nhiên `ATTN_N` câu (cùng seed cho mọi cấu hình). Hình được upload vào `attention/` (hoặc `attention/checkpoint-N/`) trên repo HF.

In [ ]:
import os, glob
from IPython.display import Image, display
ATTN_SPLIT = "test"
ATTN_N = 8
ATTN_IDS = []

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, ATTN_DIR = CKPT_INFO["model_dir"], os.path.join(CKPT_INFO["pred_dir"], "attention")
ATTN_REPO_PATH = CKPT_INFO["attn_path_in_repo"]
ids_arg = ("--ids " + " ".join(map(str, ATTN_IDS))) if ATTN_IDS else f"--n {ATTN_N}"
gt_arg = f"--gt eval_gt/{DATASET}_test_gt.json" if os.path.exists(f"eval_gt/{DATASET}_test_gt.json") else ""
!python scripts/visualize_attention.py --ckpt_dir {MODEL_DIR} --dataset {DATASET} --split {ATTN_SPLIT} {ids_arg} {gt_arg} --out_dir {ATTN_DIR}

pngs = sorted(glob.glob(os.path.join(ATTN_DIR, "*.png")))
for p in pngs:
    display(Image(filename=p))
if os.environ.get("HF_TOKEN") and HF_REPO and pngs:
    names = " ".join(os.path.basename(p) for p in pngs + [os.path.join(ATTN_DIR, "segment_mass.csv")])
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {ATTN_DIR} --path_in_repo {ATTN_REPO_PATH} --files {names}